# 11.3 Modifying data

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Many modeling projects involve solving a series of problem instances, each defined
by somewhat different `data`. We describe here AMPL's facilities for resetting parameter
values while leaving the `model` as is. They include commands for resetting `data` mode
input and for resampling random parameters, as well as the `let` command for directly
assigning new values.

**Resetting**

To `delete` the current `data` for several `model` components, without changing the current
`model` itself, use the `reset` `data` command, as in:

```ampl
reset data MINREQ, MAXREQ, amt, n_min, n_max;
```

You may then use `data` commands to read in new values for these sets and parameters.
To `delete` all `data`, type `reset` `data`.

The `update` `data` command works similarly, but does not actually `delete` any `data`
until new values are assigned. Thus if you type

```ampl
update data MINREQ, MAXREQ, amt, n_min, n_max;
```

but you only read in new values for MINREQ, amt and n_min, the previous values for
MAXREQ and n_max will remain. If instead you used `reset` `data`, MAXREQ and
n_max would be without values, and you would get an error message when you next
tried to `solve`.

**Resampling**

The `reset` `data` command also acts to resample the randomly computed parameters
described in [Section 7.6](../07/7_6_randomly_generated_parameters.ipynb). Continuing with the variant of steel4.mod introduced in
that section, if the definition of parameter avail is changed so that its value is given by
a random function:

```ampl
param avail_mean {STAGE} >= 0;
param avail_variance {STAGE} >= 0;
param avail {s in STAGE} =
   Normal(avail_mean[s], avail_variance[s]);
```

with corresponding `data`:

```ampl
param avail_mean := reheat 35 roll 40 ;
param avail_variance := reheat 5 roll 2 ;
```

then AMPL will take new samples from the Normal distribution after each `reset`
`data`. Different samples result in different solutions, and hence in different optimal
`objective` values:

```ampl
ampl: model steel4r.mod;
ampl: data steel4r.dat;
ampl: solve;
MINOS 5.5: optimal solution found.
3 iterations, objective 187632.2489
ampl: display avail;
reheat 32.3504
  roll 43.038 ;
ampl: reset data avail;
ampl: solve;
MINOS 5.5: optimal solution found.
4 iterations, objective 158882.901
ampl: display avail;
reheat 32.0306
  roll 32.6855 ;
```

Only `reset` `data` has this effect; if you issue a `reset` command then AMPL's random
number generator is `reset`, and the values of avail repeat from the beginning. ([Section 7.6](../07/7_6_randomly_generated_parameters.ipynb)
explains how to `reset` the generator's "seed" to get a different sequence of random
numbers.)

**The `let` command**

The `let` command also permits you to change particular `data` values while leaving
the `model` the same, but it is more convenient for small or easy-to-describe changes than
`reset` `data` or `update` `data`. You can use it, for example, to `solve` the diet `model` of
[Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1), trying out a series of upper bounds f_max["CHK"] on the purchases of
food CHK:

```ampl
ampl: model dietu.mod;
ampl: data dietu.dat;
ampl: solve;
MINOS 5.5: optimal solution found.

5 iterations, objective 74.27382022

ampl: let f_max["CHK"] := 11;
ampl: solve;
MINOS 5.5: optimal solution found.

1 iterations, objective 73.43818182

ampl: let f_max["CHK"] := 12;
ampl: solve;
MINOS 5.5: optimal solution found.

0 iterations, objective 73.43818182
```

Relaxing the bound to 11 reduces the cost somewhat, but further relaxation apparently
has no benefit.
An indexing expression may be given after the keyword `let`, in which case a change
is made for each member of the specified indexing set. You could use this feature to
change all upper bounds to 8:

```ampl
let {j in FOOD} f_max[j] := 8;
```

or to increase all upper bounds by 10 percent:

```ampl
let {j in FOOD} f_max[j] := 1.1 * f_max[j];
```

In general this command consists of the keyword `let`, an indexing expression if needed,
and an assignment. Any set or parameter whose declaration does not define it using an =
phrase may be specified to the left of the assignment's := operator, while to the right
may appear any appropriate expression that can currently be evaluated.

Although AMPL does not impose any restrictions on what you can change using `let`,
you should take care in changing any set or parameter that affects the indexing of other
`data`.

For example, after solving the multiperiod production problem of Figures 4-4 and 4-5,
it might be tempting to change the number of weeks T from 4 (as given in the original
`data`) to 3:

```ampl
ampl: let T := 3;
ampl: solve;
Error executing "solve" command:
error processing param avail:
        invalid subscript avail[4] discarded.
error processing param market:
        2 invalid subscripts discarded:
        market['bands',4]
        market['coils',4]
error processing param revenue:
        2 invalid subscripts discarded:
        revenue['bands',4]
        revenue['coils',4]
error processing var Sell['coils',1]:
        invalid subscript market['bands',4]
```

The problem here is that AMPL still has current `data` for 4th-week parameters such as
avail[4], which has become invalid with the change of T to 3. If you want to properly
reduce the number of weeks in the linear program while using the same `data`, you must
declare two parameters:

```ampl
param Tdata integer > 0;
param T integer <= Tdata;
```

Use 1..Tdata for indexing in the `param` declarations, while retaining 1..T for the
variables, `objective` and constraints; then you can use `let` to change T as you like.

You can also use the `let` command to change the current values of variables. This is
sometimes a convenient feature for exploring an alternative solution. For example, here

```ampl
ceil(x)            ceiling of x (next igher integer)
floor(x)           floor of x (next ower integer)
precision(x, n)    x rounded to n ignificant digits
round(x, n)        x rounded to n igits past the decimal point
round(x)           x rounded to the earest integer
trunc(x, n)        x truncated to n igits past the decimal point
trunc(x)           x truncated to an nteger (fractional part dropped)
```

**[Table 11-1](./11_3_modifying_data.ipynb#`table`-11-1):** Rounding functions.

is what happens when we `solve` for the optimal diet as above, then round the optimal
solution down to the next lowest `integer` number of packages:

```ampl
ampl: model dietu.mod; data dietu.dat; solve;
MINOS 5.5: optimal solution found.
5 iterations, objective 74.27382022
ampl: let {j in FOOD} Buy[j] := floor(Buy[j]);
ampl: display Total_Cost, n_min, Diet_Min.slack;;
Total_Cost = 70.8
:     n_min Diet_Min.slack :=
A       700        231
B1        0        580
B2        0        475
C       700        -40
CAL   16000       -640
;
```

Because we have used `let` to change the values of the variables, the `objective` and the
slacks are automatically computed from the new, rounded values. The cost has dropped
by about $3.50, but the solution is now short of the requirements for C by nearly 6 percent
and for CAL by 4 percent.

AMPL provides a variety of rounding functions that can be used in this way. They are
summarized in [Table 11-1](./11_3_modifying_data.ipynb#`table`-11-1).